# **Model pretraining - Dataset: IHC4BC**

https://www.kaggle.com/datasets/akbarnejad1991/ihc4bc-compressed

## 🧮 **Parameters**

In [1]:
# Set seed for reproducibility
SEED = 42

# Validation set relative size
N_VAL = 0.2

# Patch size
PATCH_SIZE = 160

# Augmentation
REPEATS = 1                 # Augmentations per image
AUG_PROB = 1                # Probability of performing each augmentation
MAG_ROTATION = 30           # [Degrees]
MAG_TRANSLATION = 0.2       # [Image length]
MAG_COLOR = 0.3

# Define the batch size, which is the number of samples in each batch
BATCH_SIZE = 64

# Training configuration
LEARNING_RATE = 1e-4
EPOCHS = 1000
PATIENCE = 50

# Architecture (pre-trained)
BACKBONE = "vgg16"
PRETRAINED = True
FREEZE_BACKBONE = None

# Architecture (common)
OPTIMIZER = "lion"
LABEL_SMOOTHING = 0.05

# Regularization
DROPOUT_RATE = 0.3          # Dropout probability
L1_LAMBDA = 0               # L1 penalty
L2_LAMBDA = 1e-4            # L2 penalty

## ⚙️ **Libraries Import**

In [2]:
import importlib
import subprocess
import sys
import inspect

def safe_import(path, module = None, pip_name = None):
    globals_dict = inspect.currentframe().f_back.f_globals
    pip_name = pip_name or path.split(".")[0]

    try:
        if module:
            # from path import module
            pkg = importlib.import_module(path)
            obj = getattr(pkg, module)
            globals_dict[module] = obj
        else:
            # import path
            obj = importlib.import_module(path)
            name = path.split(".")[0]
            globals_dict[name] = obj
    except ImportError:
        print(f"Installing '{pip_name}' …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        importlib.invalidate_caches()
        if module:
            # from path import module
            pkg = importlib.import_module(path)
            obj = getattr(pkg, module)
            globals_dict[module] = obj
        else:
            # import path
            obj = importlib.import_module(path)
            name = path.split(".")[0]
            globals_dict[name] = obj

In [3]:
# Import necessary libraries
import os

# Set environment variables before importing modules
os.environ['PYTHONHASHSEED'] = str(SEED)

# Suppress warnings
import warnings
warnings.simplefilter(action = "ignore", category = FutureWarning)
warnings.simplefilter(action = "ignore", category = Warning)

# Import necessary modules
import random
import numpy as np

# Set seeds for random number generators in NumPy and Python
np.random.seed(SEED)
random.seed(SEED)

# Import PyTorch
import torch
torch.manual_seed(SEED)
from torch import nn
from torchvision.transforms import v2 as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
safe_import("torchsummary", "summary")
safe_import("torchview", "draw_graph")
import torch.nn.functional as F
safe_import("ranger_adabelief", "RangerAdaBelief", pip_name = "ranger-adabelief")
safe_import("lion_pytorch", "Lion", pip_name = "lion-pytorch")

!mkdir -p "models"

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
else:
    device = torch.device("cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Device: {device}")

# Import other libraries
from PIL import Image
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
import csv

# Configure plot display settings
sns.set(font_scale = 1.4)
sns.set_style("white")
plt.rc("font", size = 14)
%matplotlib inline

PyTorch version: 2.6.0+cu124
Device: cuda


## 🥚 **Data loading**

In [4]:
subtype_names = [
    "Luminal A",
    "Luminal B",
    "HER2(+)",
    "Triple negative"
]

le = LabelEncoder()
le.fit(subtype_names)

label_to_int = {cls: i for i, cls in enumerate(le.classes_)}
int_to_label = {i: cls for i, cls in enumerate(le.classes_)}

print(label_to_int)

{'HER2(+)': 0, 'Luminal A': 1, 'Luminal B': 2, 'Triple negative': 3}


In [5]:
def get_subtype(er, pr, her2, ki67):
    # Triple negative
    if er == 0 and pr == 0 and her2 == 0:
        return "Triple negative"

    # HER2(+)
    if her2 == 1 and er == 0 and pr == 0:
        return "HER2(+)"

    # Luminal A
    if (er == 1 or pr == 1) and ki67 == 0:
        return "Luminal A"

    # Luminal B
    return "Luminal B"   # catch-all for any other ER+ subtype

In [6]:
def read_first_two_columns(path, header = True, col_names = ("file_image", " manual_annot")):
    rows = []
    with open(path, "r", newline = "", encoding = "utf-8") as f:
        reader = csv.reader(f)
        if header:
            try:
                hdr = next(reader)
            except StopIteration:
                return pd.DataFrame(columns=list(col_names))
        for r in reader:
            if not r:
                continue
            # take first two fields only; ignore the rest
            if len(r) >= 2:
                a = r[0].strip()
                b = r[1].strip()
                rows.append((a, b))
            else:
                # if there's only one column, treat second as empty
                a = r[0].strip()
                rows.append((a, ""))

    df = pd.DataFrame(rows, columns=list(col_names))
    return df

In [7]:
class IHC4BCDataset(Dataset):
    """
    Returns H&E patches with a single receptor label.
    """
    def __init__(self, base_images_path, base_labels_path,
                 receptors = ["ER", "PR", "Her2", "Ki67"],
                 label_encoder = LabelEncoder(),
                 transform = None):
        self.label_encoder = label_encoder
        self.transform = transform
        self.image_paths = []
        self.labels = []

        receptor_dfs = {}

        # Walk through label folders
        for receptor in receptors:
            receptor_csv_dir = os.path.join(base_labels_path, f"Labels_{receptor}")
            receptor_dfs[receptor] = {}

            for csv_file in os.listdir(receptor_csv_dir):
                if csv_file.endswith(".csv"):
                    patient = csv_file.replace(".csv", "")
                    df = read_first_two_columns(os.path.join(receptor_csv_dir, csv_file), header = True)
                    receptor_dfs[receptor][patient] = df

        # Walk through image folders
        for receptor in receptors:
            receptor_path = os.path.join(base_images_path, receptor)

            for patient in os.listdir(receptor_path):
                patient_path = os.path.join(receptor_path, patient)

                for subregion in os.listdir(patient_path):
                    subregion_path = os.path.join(patient_path, subregion)

                    # Find all images
                    for img_file in os.listdir(subregion_path):
                        if not img_file.lower().endswith(".jpg"):
                            continue
    
                        img_path = os.path.join(subregion_path, img_file)
                        file_image = os.path.join(patient, subregion, img_file).replace(".jpg", ".png")
    
                        # Build receptor vector
                        er = pr = her2 = ki67 = 0
    
                        for rec in receptors:
                            df = receptor_dfs[rec].get(patient)
    
                            if df is not None:
                                row = df[df['file_image'].str.contains(file_image)]
                                if len(row) > 0:
                                    annot = row.iloc[0][' manual_annot']
                                    if annot.strip() == "a":
                                        if rec == "ER": er = 1
                                        if rec == "PR": pr = 1
                                        if rec == "Her2": her2 = 1
                                        if rec == "Ki67": ki67 = 1
    
                        subtype = label_encoder.transform([get_subtype(er, pr, her2, ki67)])[0]
    
                        self.image_paths.append(img_path)
                        self.labels.append(subtype)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx])
        if self.transform:
            image = self.transform(image)
            
        label = torch.tensor(self.labels[idx], dtype = torch.long)
        return image, label

In [8]:
# Build dataset
dataset = IHC4BCDataset(
    "../input/ihc4bc-compressed/IHC4BC_Compressed/Images/HandE",
    "../input/ihc4bc-compressed/IHC4BC_Compressed/Labels",
    receptors = ["ER", "PR", "Her2", "Ki67"],
    label_encoder = le,
    transform = transforms.Compose([transforms.Resize((PATCH_SIZE, PATCH_SIZE)), transforms.ToTensor()])
)

img, labels = dataset[0]
print(img.shape, labels)  # labels will be a tensor of shape [4] with values 0-3

torch.Size([3, 160, 160]) tensor(3)


In [9]:
# Split ratios
train_ratio = 1 - N_VAL
val_ratio = N_VAL
dataset_size = len(dataset)
train_size = int(train_ratio * dataset_size)
val_size = dataset_size - train_size

# Random split
train_ds_base, val_ds_base = random_split(dataset, [train_size, val_size])

## 💪 **Image augmentation**

In [10]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class ImageNetNormalize:
    """
    Normalize RGB channels with ImageNet mean/std.
    """
    def __init__(self, mean = IMAGENET_MEAN, std = IMAGENET_STD):
        self.mean = torch.tensor(mean, dtype = torch.float32).reshape(3, 1, 1)
        self.std = torch.tensor(std, dtype = torch.float32).reshape(3, 1, 1)

    def __call__(self, image):
        C = image.shape[0]

        if C == 1:
            # grayscale
            gray = image
            rgb = gray.repeat(3, 1, 1)
        elif C == 3:
            # RGB
            rgb = image
        else:
            raise ValueError("Input must have 1 or 3 channels")

        # Normalize only RGB
        rgb_normalized = (rgb - self.mean) / self.std

        return rgb_normalized

In [11]:
class AugmentedDataset(torch.utils.data.Dataset):
    """
    Dataset wrapper that applies augmentation transforms to images.
    """
    def __init__(self, base_dataset, repeats = 1, transform = None):
        self.base_dataset = base_dataset
        self.repeats = repeats
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset) * self.repeats

    def __getitem__(self, idx):
        base_idx = idx % len(self.base_dataset)

        sample = self.base_dataset[base_idx]

        # Training/validation: (image, label)
        image, label = sample
        if self.transform:
            image = self.transform(image)
        return image, label

In [12]:
# Geometric ops
def rotate(image, mag):
    return transforms.functional.rotate(
        image, angle = float(mag),
        interpolation = transforms.InterpolationMode.BILINEAR
    )

def translate_x(image, mag):
    return transforms.functional.affine(
        image, angle  = 0,
        translate = [int(mag * image.shape[-1]), 0],
        scale = 1.0, shear = 0,
        interpolation = transforms.InterpolationMode.BILINEAR
    )

def translate_y(image, mag):
    return transforms.functional.affine(
        image, angle = 0,
        translate = [0, int(mag * image.shape[-2])],
        scale = 1.0, shear = 0,
        interpolation = transforms.InterpolationMode.BILINEAR
    )

# Color ops
def adjust_brightness(image, mag):
    return transforms.functional.adjust_brightness(image, 1.0 + float(mag))

def adjust_contrast(image, mag):
    return transforms.functional.adjust_contrast(image, 1.0 + float(mag))

class RandAugment:
    def __init__(self):
        self.rotation_ops = [
            ('rotate', rotate),
        ]

        self.translation_ops = [
            ('translate_x', translate_x),
            ('translate_y', translate_y),
        ]

        self.color_ops = [
            ('brightness', adjust_brightness),
            ('contrast', adjust_contrast),
        ]

In [13]:
class ImageRandAugment:
    """
    RandAugment for images.
    """
    def __init__(self,
                 aug_prob = 0.5,
                 mag_rotation = 30,
                 mag_translation = 0.1,
                 mag_color = 0.2):
        self.rand_augment = RandAugment()
        self.aug_prob = aug_prob
        self.mag_rotation = mag_rotation
        self.mag_translation = mag_translation
        self.mag_color = mag_color

        # Base geometric transforms (flips)
        self.base_geometric = transforms.Compose([
            transforms.RandomHorizontalFlip(p = 0.5),
            transforms.RandomVerticalFlip(p = 0.5),
        ])

    def _augment(self, image, ops, mag):
        for _, op in ops:
            if random.random() <= self.aug_prob:
                mag = random.uniform(-mag, mag)
                image = op(image, mag)
        return image

    def __call__(self, image):
        # Apply flips
        image = self.base_geometric(image)

        # Apply geometric RandAugment
        image = self._augment(image, self.rand_augment.rotation_ops,    self.mag_rotation)
        image = self._augment(image, self.rand_augment.translation_ops, self.mag_translation)

        # Apply color RandAugment
        image = self._augment(image, self.rand_augment.color_ops, self.mag_color)

        return image

In [14]:
# Create transform
train_transform = ImageRandAugment(
    aug_prob = AUG_PROB,
    mag_rotation = MAG_ROTATION,
    mag_translation = MAG_TRANSLATION,
    mag_color = MAG_COLOR
)

normalize_transform = ImageNetNormalize()

In [15]:
# Apply augmentation + normalization
train_ds_aug = AugmentedDataset(
    train_ds_base,
    repeats = REPEATS,
    transform = transforms.Compose([
        train_transform,
        normalize_transform
    ])
)

# Apply only normalization
val_ds_aug = AugmentedDataset(
    val_ds_base,
    repeats = 1,
    transform = normalize_transform
)

print(f"Training:   {len(train_ds_aug):>6,} samples (with RandAugment)")
print(f"Validation: {len(val_ds_aug):>6,} samples (no augmentation)")

Training:   74,001 samples (with RandAugment)
Validation: 18,501 samples (no augmentation)


In [16]:
labels = torch.tensor(train_ds_aug.base_dataset.dataset.labels, dtype = torch.long)

# Compute class counts
num_classes = int(labels.max().item() + 1)
class_counts = torch.bincount(labels, minlength = num_classes)

# Compute soft class weights
class_weights = 1.0 / torch.sqrt(class_counts.float())
sample_weights = class_weights[labels]
sample_weights = sample_weights.repeat(train_ds_aug.repeats)

# Build sampler
train_sampler = WeightedRandomSampler(
    weights = sample_weights,
    num_samples = len(sample_weights),
    replacement = True
)

In [17]:
def make_loader(ds, batch_size, shuffle, drop_last, sampler = None):
   # Determine optimal number of worker processes for data loading
    cpu_cores = os.cpu_count() or 2
    num_workers = max(2, min(4, cpu_cores))
    
    return DataLoader(
        ds,
        batch_size = batch_size,
        shuffle = (sampler is None and shuffle),
        sampler = sampler,
        drop_last = drop_last,
        num_workers = num_workers,
        pin_memory = True,  # Faster GPU transfer
        pin_memory_device = "cuda" if torch.cuda.is_available() else "",
        prefetch_factor = 4,  # Load 4 batches ahead
    )

# Create data loaders
train_loader = make_loader(train_ds_aug, batch_size = BATCH_SIZE, shuffle = True,  drop_last = False, sampler = train_sampler)
val_loader   = make_loader(val_ds_aug,   batch_size = BATCH_SIZE, shuffle = False, drop_last = False)

## 🐤 **Model building**

In [18]:
class PretrainCNN(nn.Module):
    """
    Pretrained backbone + mask-guided attention.
    """
    def __init__(
        self,
        num_classes = 4,
        backbone = "efficientnet_b0",
        pretrained = True,
        custom_weights = None,
        freeze_backbone = "partial",
        dropout_rate = 0.2
    ):
        super().__init__()

        # ------------------------------------------------------------
        # 1) Backbone selection
        # ------------------------------------------------------------
        backbone_dict = {
            "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1),
            "resnet18":        (models.resnet18,        models.ResNet18_Weights.IMAGENET1K_V1),
            "resnet34":        (models.resnet34,        models.ResNet34_Weights.IMAGENET1K_V1),
            "resnet50":        (models.resnet50,        models.ResNet50_Weights.IMAGENET1K_V2),
            "resnet101":       (models.resnet101,       models.ResNet101_Weights.IMAGENET1K_V2),
            "vgg16":           (models.vgg16_bn,        models.VGG16_BN_Weights.IMAGENET1K_V1),
        }

        if backbone not in backbone_dict:
            raise ValueError(f"Unsupported backbone '{backbone}'")

        model_func, weights = backbone_dict[backbone]
        backbone_model = model_func(weights = weights if pretrained else None)

        # ------------------------------------------------------------
        # 2) Feature extractor
        # ------------------------------------------------------------
        if backbone.startswith("efficientnet"):
            self.feature_extractor = nn.Sequential(backbone_model.features, backbone_model.avgpool)
            self.feat_channels = backbone_model.classifier[1].in_features

        elif backbone.startswith("resnet"):
            self.feature_extractor = nn.Sequential(
                backbone_model.conv1,
                backbone_model.bn1,
                backbone_model.relu,
                backbone_model.maxpool,
                backbone_model.layer1,
                backbone_model.layer2,
                backbone_model.layer3,
                backbone_model.layer4
            )
            self.feat_channels = backbone_model.fc.in_features

        elif backbone == "vgg16":
            self.feature_extractor = backbone_model.features
            self.feat_channels = 512  # VGG16 features output 512 channels

        # ------------------------------------------------------------
        # 3) Classification head
        # ------------------------------------------------------------
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(self.feat_channels, 256),
            nn.ReLU(inplace = True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 64),
            nn.ReLU(inplace = True),
            nn.Dropout(dropout_rate),
            nn.Linear(64, num_classes),
        )

        # ------------------------------------------------------------
        # 4) Load custom pretrained weights
        # ------------------------------------------------------------
        if custom_weights is not None:
            self.load_state_dict(custom_weights)

        # ------------------------------------------------------------
        # 5) Freeze backbone parameters
        # ------------------------------------------------------------
        if freeze_backbone == "all":
            for p in backbone_model.parameters():
                p.requires_grad = False

        elif freeze_backbone == "partial":
            if backbone.startswith("efficientnet"):
                # Freeze early blocks 0–3 (like ResNet low-level freeze)
                for name, module in backbone_model.features.named_children():
                    if int(name) <= 3:
                        for p in module.parameters():
                            p.requires_grad = False

            elif backbone.startswith("resnet"):
                # Freeze low-level blocks: conv1, bn1, layer1, layer2
                for name, p in backbone_model.named_parameters():
                    if any(name.startswith(pref) for pref in ["conv1", "bn1", "layer1", "layer2", "layer3"]):
                        p.requires_grad = False

            elif backbone == "vgg16":
                # Freeze first half of VGG features (very generic filters)
                num_features = len(backbone_model.features)
                for i, m in enumerate(backbone_model.features):
                    if i < num_features // 2:
                        for p in m.parameters():
                            p.requires_grad = False

        # ------------------------------------------------------------
        # 6) Re-initialize classifier
        # ------------------------------------------------------------
        self._init_weights()

    def _init_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, x):
        """
        x: tensor (B, C, H, W)
        """
        # Backbone conv features
        feat = self.feature_extractor(x)   # (B, feat_channels, Hf, Wf)

        # Average pooling
        pooled = F.adaptive_avg_pool2d(feat, (1, 1)).view(x.size(0), -1)

        logits = self.classifier(pooled)
        return logits

In [19]:
model = PretrainCNN(
    num_classes = 4,
    backbone = BACKBONE,
    pretrained = PRETRAINED,
    freeze_backbone = FREEZE_BACKBONE,
    dropout_rate = DROPOUT_RATE
).to(device)

Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth
100%|██████████| 528M/528M [00:02<00:00, 223MB/s] 


In [20]:
class FocalLoss(nn.Module):
    """
    Focal Loss for addressing class imbalance.
    Focuses on hard-to-classify examples.
    """
    def __init__(self, alpha = None, gamma = 2.0, label_smoothing = 0.0, reduction = "mean"):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs, targets):
        # Get cross entropy loss
        ce_loss = nn.functional.cross_entropy(
            inputs,
            targets,
            reduction = "none",
            label_smoothing = self.label_smoothing
        )

        # Calculate focal weight
        pt = torch.exp(-ce_loss)
        focal_weight = (1 - pt) ** self.gamma
        focal_loss = focal_weight * ce_loss

        # Apply class weights if provided
        if self.alpha is not None:
            if self.alpha.device != inputs.device:
                self.alpha = self.alpha.to(inputs.device)
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss

        # Apply reduction
        if self.reduction == "mean":
            return focal_loss.mean()
        elif self.reduction == "sum":
            return focal_loss.sum()
        else:
            return focal_loss

In [21]:
# Enable mixed precision training for GPU acceleration
scaler = torch.amp.GradScaler(enabled = (device.type == "cuda"))

criterion = FocalLoss(alpha = None, gamma = 2.0, label_smoothing = LABEL_SMOOTHING)

In [22]:
optimizer_dict = {
    "adam":   torch.optim.Adam( model.parameters(), lr = LEARNING_RATE, weight_decay = L2_LAMBDA),
    "adamW":  torch.optim.AdamW(model.parameters(), lr = LEARNING_RATE, weight_decay = L2_LAMBDA),
    "ranger": RangerAdaBelief(  model.parameters(), lr = LEARNING_RATE, weight_decay = L2_LAMBDA),
    "lion":   Lion(             model.parameters(), lr = LEARNING_RATE, weight_decay = L2_LAMBDA),
}

# Loss function selection
try:
    optimizer = optimizer_dict[OPTIMIZER]
except KeyError:
    raise ValueError(f"Optimizer '{OPTIMIZER}' not recognized")

Ranger optimizer loaded. 
Gradient Centralization usage = True
GC applied to both conv and fc layers


## 🦖 **Pre-training**

In [23]:
def train_one_epoch(model, train_loader, criterion, optimizer, scaler, device, l1_lambda = 0, l2_lambda = 0):
    """
    Perform one complete training epoch through the entire training dataset.
    """
    model.train()  # Set model to training mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Iterate through training batches
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        # Move data to device (GPU/CPU)
        inputs, targets = inputs.to(device), targets.to(device)

        # Clear gradients from previous step
        optimizer.zero_grad(set_to_none = True)

        # Forward pass with mixed precision (if CUDA available)
        with torch.amp.autocast(device_type = device.type, enabled = (device.type == "cuda")):
            logits = model(inputs)
            loss = criterion(logits, targets)

            # Add L1 and L2 regularization
            l1_norm = sum(p.abs().sum() for p in model.parameters())
            l2_norm = sum(p.pow(2).sum() for p in model.parameters())
            loss = loss + l1_lambda * l1_norm + l2_lambda * l2_norm

        # Backward pass with gradient scaling
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Accumulate metrics
        running_loss += loss.item() * inputs.size(0)
        predictions = torch.argmax(logits, dim = 1)
        all_predictions.append(predictions.cpu().numpy())
        all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average = "weighted"
    )

    return epoch_loss, epoch_f1

In [24]:
def validate_one_epoch(model, val_loader, criterion, device):
    """
    Perform one complete validation epoch through the entire validation dataset.
    """
    model.eval()  # Set model to evaluation mode

    running_loss = 0.0
    all_predictions = []
    all_targets = []

    # Disable gradient computation for validation
    with torch.no_grad():
        for inputs, targets in val_loader:
            # Move data to device
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass with mixed precision (if CUDA available)
            with torch.amp.autocast(device_type = device.type, enabled = (device.type == "cuda")):
                logits = model(inputs)
                loss = criterion(logits, targets)

            # Accumulate metrics
            running_loss += loss.item() * inputs.size(0)
            predictions = torch.argmax(logits, dim = 1)
            all_predictions.append(predictions.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Calculate epoch metrics
    epoch_loss = running_loss / len(val_loader.dataset)
    epoch_f1 = f1_score(
        np.concatenate(all_targets),
        np.concatenate(all_predictions),
        average = "weighted"
    )

    return epoch_loss, epoch_f1

In [25]:
def fit(model, train_loader, val_loader, epochs, criterion, optimizer, scaler, device,
        l1_lambda = 0, l2_lambda = 0, patience = 0, evaluation_metric = "val_f1", mode = "max",
        restore_best_weights = True, verbose = 10, experiment_name = ""):
    """
    Train the neural network model on the training data and validate on the validation data.
    """
    # Initialize metrics tracking
    training_history = {
        "train_loss": [], "val_loss": [],
        "train_f1": [], "val_f1": []
    }

    # Configure early stopping if patience is set
    if patience > 0:
        patience_counter = 0
        best_metric = float("-inf") if mode == "max" else float("inf")
        best_epoch = 0

    print(f"Training {epochs} epochs...")

    try:
        # Main training loop: iterate through epochs
        for epoch in range(1, epochs + 1):

            # Forward pass through training data, compute gradients, update weights
            train_loss, train_f1 = train_one_epoch(
                model, train_loader, criterion, optimizer, scaler, device, l1_lambda, l2_lambda
            )

            # Evaluate model on validation data without updating weights
            val_loss, val_f1 = validate_one_epoch(
                model, val_loader, criterion, device
            )

            # Store metrics for plotting and analysis
            training_history['train_loss'].append(train_loss)
            training_history['val_loss'].append(val_loss)
            training_history['train_f1'].append(train_f1)
            training_history['val_f1'].append(val_f1)

            # Print progress every N epochs or on first epoch
            if verbose > 0:
                if epoch % verbose == 0 or epoch == 1:
                    print(f"Epoch {epoch:3d}/{epochs} | "
                        f"Train: Loss={train_loss:.4f}, F1 Score={train_f1:.4f} | "
                        f"Val: Loss={val_loss:.4f}, F1 Score={val_f1:.4f}")

            # Early stopping logic: monitor metric and save best model
            if patience > 0:
                current_metric = training_history[evaluation_metric][-1]
                is_improvement = (current_metric > best_metric) if mode == "max" else (current_metric < best_metric)

                if is_improvement:
                    best_metric = current_metric
                    best_epoch = epoch
                    torch.save(model.state_dict(), "models/" + experiment_name + "_model.pt")
                    patience_counter = 0
                else:
                    patience_counter += 1
                    if patience_counter >= patience:
                        print(f"Early stopping triggered after {epoch} epochs.")
                        break

    except KeyboardInterrupt:
        print("Training interrupted by user. Saving current model...")

        # save model even if interrupted
        torch.save(model.state_dict(), "models/" + experiment_name + "_interrupted.pt")

    # Restore best model weights if early stopping was used
    if restore_best_weights and patience > 0:
        model.load_state_dict(torch.load("models/" + experiment_name + "_model.pt"))
        print(f"Best model restored from epoch {best_epoch} with {evaluation_metric} {best_metric:.4f}")

    # Save final model if no early stopping
    if patience == 0:
        torch.save(model.state_dict(), "models/" + experiment_name + "_model.pt")

    return model, training_history

In [26]:
%%time
# Train model and track training history
model, history = fit(
    model = model,
    train_loader = train_loader,
    val_loader = val_loader,
    epochs = EPOCHS,
    criterion = criterion,
    optimizer = optimizer,
    scaler = scaler,
    device = device,
    verbose = 1,
    patience = PATIENCE
)

Training 1000 epochs...
Epoch   1/1000 | Train: Loss=0.1662, F1 Score=0.8690 | Val: Loss=0.5326, F1 Score=0.6738
Epoch   2/1000 | Train: Loss=0.1023, F1 Score=0.9243 | Val: Loss=0.1993, F1 Score=0.8287
Epoch   3/1000 | Train: Loss=0.0962, F1 Score=0.9306 | Val: Loss=0.1102, F1 Score=0.8864
Epoch   4/1000 | Train: Loss=0.0905, F1 Score=0.9357 | Val: Loss=0.0771, F1 Score=0.9323
Epoch   5/1000 | Train: Loss=0.0877, F1 Score=0.9384 | Val: Loss=0.1100, F1 Score=0.9133
Training interrupted by user. Saving current model...
Best model restored from epoch 4 with val_f1 0.9323
CPU times: user 35min 59s, sys: 4min 34s, total: 40min 33s
Wall time: 1h 46s


In [27]:
weights_path = "/kaggle/working/" + BACKBONE + "_pretrained.pt"
torch.save(model.state_dict(), weights_path)
print(f"Model weights saved to {weights_path}")

Model weights saved to /kaggle/working/vgg16_pretrained.pt
